In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv("./course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data Prep

In [4]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

imputing missing values

In [8]:
categoricals = df.dtypes[df.dtypes == 'str'].index.tolist()
numericals = df.dtypes[df.dtypes != 'str'].index.tolist()
numericals.remove('converted')

for c in categoricals:
    df[c] = df[c].fillna('NA')

for n in numericals:
    df[n] = df[n].fillna(0)

splitting the data

In [9]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

Q1 - ROC AUC feature importance

In [10]:
from sklearn.metrics import roc_auc_score
for n in numericals:
    auc = roc_auc_score(df_train['converted'], df_train[n])
    print(f"{n}: {abs(auc):.3f}")

annual_income: 0.608
number_of_courses_viewed: 0.723
interaction_count: 0.765
lead_score: 0.788


Q2 - training the model

In [11]:
# one hot encoding
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=False)

train_dicts = df_train[categoricals + numericals].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)
y_train = df_train['converted'].values

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)
model.predict_proba